## ✅ Conversion Validation Report — PASS

- **Workflow:** 02_join_summarize
- **Tools converted:** 5 (Low complexity, 1 joins, 0 formulas)
- **Self-correction iterations used:** 1 of 3

| Iteration | Mode | Result | Failures | Warnings | Action taken |
|---|---|---|---|---|---|
| 1 | standard | PASS | 0 | 0 | validated clean — exported |

# 02_join_summarize
Auto-generated by the Alteryx → Databricks Converter Engine
Source: 02_join_summarize.yxmd | Mode: standard
Altryx Accelerator 1.1.0
Complexity: Low (5 tools, 1 joins, 0 formulas)

In [ ]:
# SOURCE_TABLES: map tool ID, annotation, or path substring to a Unity Catalog table.
SOURCE_TABLES = {'1': 'main.sales.transactions'}
SOURCE_LABELS = {'1': ['Transactions table', 'main.sales.transactions']}
WORKFLOW_CONSTANTS = {}
def _resolve_source(tool_id):
    if SOURCE_TABLES.get(tool_id):
        return SOURCE_TABLES[tool_id]
    for key, value in SOURCE_TABLES.items():
        if value and (str(key) == tool_id or any(str(key).lower() in label.lower() for label in SOURCE_LABELS[tool_id])):
            return value
    raise ValueError('Map input ' + tool_id + ' in the SOURCE_TABLES configuration cell before running.')

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Input Sources

In [ ]:
df_transactions_table = spark.table(_resolve_source('1'))  # Tool 1: InputData
# Tool 2: TextInput
_data_2 = [
    ('10', 'Enterprise'),
    ('20', 'SMB'),
    ('30', 'Consumer'),
]
df_customer_segments = spark.createDataFrame(_data_2, ['CustomerID', 'Segment'])
df_customer_segments = df_customer_segments.withColumn("CustomerID", F.col("CustomerID").cast("long"))

## Transformations

In [ ]:
# Tool 3: Join
_left_3 = df_transactions_table.select(*[F.col('`' + c.replace('`', '``') + '`').alias('Left_' + c) for c in df_transactions_table.columns])
_right_3 = df_customer_segments.select(*[F.col('`' + c.replace('`', '``') + '`').alias('Right_' + c) for c in df_customer_segments.columns])
df_3_joined = _left_3.join(F.broadcast(_right_3), (_left_3['Left_CustomerID'] == _right_3['Right_CustomerID']), "inner")
_spec_3 = {s['field']: s for s in [{'field': 'Right_CustomerID', 'selected': 'False', 'rename': '', 'type': ''}, {'field': '*Unknown', 'selected': 'True', 'rename': '', 'type': ''}]}
_projection_3, _used_3 = [], set()
for _c in df_3_joined.columns:
    _raw = _c.split('_', 1)[1]
    _s = _spec_3.get(_c, _spec_3.get(_raw, {}))
    if _s.get('selected', 'True') == 'False':
        continue
    _name = _s.get('rename') or _raw
    if _name in _used_3:
        _name = _c
    _used_3.add(_name)
    _projection_3.append(F.col('`' + _c.replace('`', '``') + '`').alias(_name))
df_3_joined = df_3_joined.select(*_projection_3)
# Tool 4: Summarize
df_revenue_by_region_and_segment = df_3_joined.groupBy("Region", "Segment").agg(
    F.sum("Revenue").alias("TotalRevenue"),
    F.count("TxnID").alias("TxnCount")
)

## Output

In [ ]:
# Tool 5: OutputData -> main.gold.revenue_by_region_segment
df_5 = df_revenue_by_region_and_segment
# df_5.write.format("delta").mode("overwrite").saveAsTable("main.gold.revenue_by_region_segment")
df_5.createOrReplaceTempView("revenue_by_region_segment")

## Validation

In [ ]:
# Row count sanity check
# Uncomment and set df_final to your final output DataFrame
# row_count = df_final.count()
# print(f"Final row count: {row_count:,}")
# assert row_count > 0, "Output DataFrame is empty!"
# df_final.limit(5).display()